# XLM-RoBERTa Fine-Tuning ile Duygu Sınıflandırması (Sentiment Analysis)

Bu notebook'ta HuggingFace üzerinde bulunan 100 dilli devasa **`xlm-roberta-base`** modelini (Facebook AI) alıp, kendi Türkçe veri setimiz üzerinde (Fine-Tuning) eğiteceğiz.

XLM-RoBERTa, klasik BERT'e göre çok daha geniş bir veri setiyle ve daha optimize edilmiş bir mimariyle (Robustly Optimized BERT Approach) eğitildiği için genellikle duygu analizinde daha yüksek F1-Score sonuçları vermektedir.

### Kullanılan Teknikler
| Teknik | Açıklama |
|---|---|
| **Transfer Learning** | XLM-RoBERTa modelini kendi verimize uyarlıyoruz |
| **Fine-Tuning** | Modelin sınıflandırma katmanlarını kendi verimizle yeniden eğitiyoruz |
| **HuggingFace Trainer API** | Eğitim döngüsünü otomatikleştiriyoruz |
| **GPU Hızlandırma** | RTX 3060 GPU ile eğitimi hızlandırıyoruz |

## Bölüm 1 — Ortam Kurulumu

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)

# Tekrarlanabilirlik
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Kullanılan Cihaz: {device.upper()}")


## Bölüm 2 — Veri Setinin Yüklenmesi ve Hazırlanması

In [ ]:
df = pd.read_csv('data/processed/reviews_cleaned.csv')

# NaN'ları temizle
df = df.dropna(subset=['cleaned_text', 'label'])

# Undersampling (Sınıf Dengeleme) - Hızlı eğitim için her sınıftan 8000 örnek alalım (Veya verinizin boyutuna göre ayarlayın)
min_class_size = df['label'].value_counts().min()
# Daha hızlı eğitim için min_class_size'ı 5000 ile sınırlayabilirsiniz: min_class_size = min(min_class_size, 5000)

df_balanced = df.groupby('sentiment').apply(lambda x: x.sample(min_class_size, random_state=SEED)).reset_index(drop=True)

print("Dengelenmiş Veri Sınıf Dağılımı:")
print(df_balanced['label'].value_counts())

# Etiketleri sayısallaştırma
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

df_balanced['label'] = df_balanced['label'].map(label2id)

# Train-Test Split (%80 Eğitim, %20 Test)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced['cleaned_text'].tolist(),
    df_balanced['label'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df_balanced['label'].tolist()
)

print(f"Eğitim seti boyutu: {len(train_texts)}")
print(f"Test seti boyutu: {len(test_texts)}")


## Bölüm 3 — Tokenizasyon ve Dataset Formatı

In [ ]:
model_name = "xlm-roberta-base"

# XLM-RoBERTa Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)


## Bölüm 4 — Modelin Yüklenmesi ve Eğitim (Fine-Tuning)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir='./models/xlm-roberta-results',
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./models/logs',     
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)


In [ ]:
# Eğitimi Başlat (Çalıştırmak için yorum satırını kaldırın)
trainer.train()


## Bölüm 5 — Modeli Kaydetme ve Değerlendirme

In [ ]:
# Modeli Kaydet
trainer.save_model("./models/xlm-roberta-sentiment")
tokenizer.save_pretrained("./models/xlm-roberta-sentiment")
print("Eğitim tamamlandığında model ./models/xlm-roberta-sentiment dizinine kaydedilecektir.")


In [ ]:
# Test seti üzerinde değerlendirme yap
results = trainer.evaluate()
print("Test Seti Sonuçları:", results)
